# Weka gateway benchmarking

Generate output JSON that we can then aggregate, plot, and do statistics on.

## Methodology

This test uses the Butler to get handles for a large number of queries from one or more datasets.  Acquiring the query handles does not pull the actual query result data.  We then select a much smaller sample of those queries to retrieve the result data.

For each query within a given dataset, we measure the size of the HTTP object (referenced with a signed URL via the Weka-S3 gateway at USDF) to retrieve, and fetch the object.  The fetch is performed with `curl`, because we believe that `curl` redirected to `/dev/null` will be much faster than pulling an HTTP object with a Python library.

After all objects have been fetched, the retrieval is (optionally) repeated, to give us insight into the cache size and replacement policy.

This process is then repeated for each dataset.

Statistics (mean, median, population variance, population standard deviation, and coefficient of variation) are collected for each of: object size, measured pull speed, reported pull speed, and ratio of reported speed to measured speed.

If `recheck` is enabled, statistics will be generated for the ratio of reported speeds and of measured speeds between a first and second fetch, to determine cache effects.

The results are then persisted into a file in a shared directory, so we can eventually aggregate over those files, allowing us to test performance of many retrievals in parallel via the `mobu` testing framework.

In [ ]:
# If you're running from Mobu, comment out the import, and paste the contents
# of weka_helpers.py below that.  That's because Mobu doesn't run a Python
# file as such--it runs a (possibly large) Python string.  Thus any import
# that isn't on the system path is going to fail, because it simply doesn't
# exist on the pod running Mobu code.
#
# The version with inlined weka_helpers.py is on the "mobu" branch.  It's
# your job to update that branch whenever you update weka_helpers.py on
# "main".

from weka_helpers import (
    Benchmarker,
    Dataset,
    DatasetList,
)

In [ ]:
# Imports
import json
from pathlib import Path

## Set up input parameters

In [ ]:
# Set up report output directory
output_dir = Path("/rubin/bench/weka/current")  # Only on data-int/dev

# Debugging output?
debug = True

# Send results to sasquatch?
send_to_sasquatch = True

potential_datasets = {
    "raw@usdf": Dataset(
        butler_config="dp2",
        butler_collections="dp2",
        display_name="raw@usdf",
        dataset_name="raw",  # 3.6GB-ish, consistent
        count=5,
        limit=10_000,
        location="USDF",
        recheck=True,
    ),
    "deep_coadd@google": Dataset(
        butler_config="dp2",
        butler_collections="dp2",
        display_name="deep_coadd@google",
        dataset_name="deep_coadd",  # 32MB-ish, consistent
        count=100,
        limit=10_000,
        location="Google",
        recheck=True,
    ),
    "visit_image@usdf": Dataset(
        butler_config="dp2",
        butler_collections="dp2",
        display_name="visit_image@usdf",
        dataset_name="visit_image",  # 30MB-ish, consistent
        count=100,
        limit=10_000,
        location="USDF",
        recheck=True,
    ),
    "run_provenance@usdf": Dataset(
        butler_config="dp2",
        butler_collections="dp2",
        display_name="run_provenance@usdf",
        dataset_name="run_provenance",  # 1GB-ish, variable
        count=10,
        limit=56,
        location="USDF",
        recheck=True,
    ),
}

# Define our input data parameters
dataset_list: DatasetList = [
    potential_datasets[x] for x in ["deep_coadd@google", "visit_image@usdf"]
]

# anything else with a "+" on this page is also hosted at USDF: https://rubinobs.atlassian.net/wiki/x/CgCveg
# (anything without a +/pencil is on GCS)

## Fetch results

In [ ]:
benchmarker = Benchmarker(
    dataset_list=dataset_list,
    output_dir=output_dir,
    debug=debug,
    send_to_sasquatch=send_to_sasquatch,
)
res = await benchmarker.generate_results()

# Prettify result
print(json.dumps(json.loads(res.model_dump_json()), sort_keys=True, indent=2))